In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, regexp_extract, col

# Define source_path using wildcards (*) to get year and month dinamically 
source_path = "abfss://landing@stroutemindeuskadidev.dfs.core.windows.net/weather/historical_readings/*/*/*.xml"
delta_path = "abfss://bronze@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/wheather_history/data"
deltaTable = "dbw_routemind_euskadi_dev.bronze.weather_history"


df_weather_hist_raw = spark.read \
    .format("text") \
    .option("wholetext", "true") \
    .load(source_path)


In [0]:

df_weather_hist_bronze = df_weather_hist_raw.select(
    input_file_name().alias("file_path"),
    regexp_extract(input_file_name(), r"/([^/]+)\.xml$", 1).alias("sensor_id"), 
    col("value").alias("raw_xml"),
    current_timestamp().alias("ingestion_timestamp")
)

In [0]:
df_weather_hist_bronze.printSchema()

In [0]:

df_weather_hist_bronze.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(deltaTable)


print(f"APPEND completed on {deltaTable}. rows processed: {df_weather_hist_bronze.count()}")


In [0]:
%sql
select count(*) from dbw_routemind_euskadi_dev.bronze.air_quality_history


In [0]:
%sql
select * from dbw_routemind_euskadi_dev.bronze.cultural_places 
LIMIT 20 